In [2]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 8.0 MB/s eta 0:00:00


In [12]:
# Setup
from groq import Groq
import os
import getpass

# Secure API Key Input
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL_NAME = "llama-3.1-8b-instant"

In [18]:
MODEL_CONFIG = {
    "technical": {
        "model": MODEL_NAME,
        "system_prompt": (
            "You are a Technical Support Expert. "
            "Be precise, rigorous, and code-focused. "
            "Provide debugging steps and example fixes."
        ),
        "temperature": 0.7
    },
    "billing": {
        "model": MODEL_NAME,
        "system_prompt": (
            "You are a Billing Support Expert. "
            "Be empathetic, policy-aware, and professional. "
            "Explain refund policies clearly."
        ),
        "temperature": 0.7
    },
    "general": {
        "model": MODEL_NAME,
        "system_prompt": "You are a helpful general assistant.",
        "temperature": 0.7
    },
    "tool": {}
}


# Router

def route_prompt(user_input):

    routing_prompt = (
        "You are a strict intent classifier.\n\n"
        "Classify the user text into EXACTLY ONE of these categories:\n"
        "- technical: programming errors, bugs, debugging, code issues\n"
        "- billing: charges, refunds, payments, subscriptions\n"
        "- tool: requests for real-time data like prices, current time, live information\n"
        "- general: casual conversation or general knowledge\n\n"
        "Return ONLY the category name.\n\n"
        f"Text: {user_input}"
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {"role": "user", "content": routing_prompt}
        ]
    )

    return response.choices[0].message.content.strip().lower()

# Tool Expert

def fetch_bitcoin_price():
    return "The current price of Bitcoin is approximately $50,000 (mock data)."


# Orchestrator

def process_request(user_input):

    category = route_prompt(user_input)
    print("Router Decision:", category)

    if category == "tool":
        if "bitcoin" in user_input.lower():
            return fetch_bitcoin_price()
        else:
            return "Tool request not supported."

    if category not in MODEL_CONFIG:
        category = "general"

    config = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model=config["model"],
        temperature=config["temperature"],
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [21]:
#Test

print(process_request("My python script is throwing an IndexError."))
print("----------------------------------------------------------------------")
print(process_request("I was charged twice for my subscription."))
print("----------------------------------------------------------------------")
print(process_request("What is the current price of Bitcoin?"))
print("----------------------------------------------------------------------")
print(process_request("Hi, can you tell me what your company does?"))

Router Decision: technical
**IndexError: What it means and how to fix it**

An `IndexError` in Python occurs when you try to access an element in a sequence (such as a list, tuple, or string) at an index that is out of range. This means you're trying to access an element that doesn't exist.

**Example:**

```python
my_list = [1, 2, 3]
print(my_list[3])  # IndexError: list index out of range
```

**Debugging steps:**

1. **Check your indexing**: Make sure you're not trying to access an index that's greater than or equal to the length of the sequence.
2. **Check for empty sequences**: If your sequence is empty, accessing any index will raise an `IndexError`.
3. **Check for None values**: If your sequence contains `None` values, accessing an index that contains `None` will raise an `IndexError`.

**Example fixes:**

1. **Check the length of the sequence**:
```python
my_list = [1, 2, 3]
if len(my_list) > 3:
    print(my_list[3])
else:
    print("Index out of range")
```

2. **Use try-excep